In [6]:
%load_ext autoreload
%autoreload 2
import sys, time
sys.path.insert(0, "..")
from src import sql_araci

print("DB:", sql_araci.ayarlar.DB_YOLU, "| var mı:", sql_araci.ayarlar.DB_YOLU.exists())
print("oyun_adlari:", len(sql_araci.oyun_adlari), "satır\n")

testler = [
    ("oyun_bul", lambda: sql_araci.oyun_bul_araci("sims 4")),
    ("normal sorgu", lambda: sql_araci.sql_calistir("SELECT ad, olumlu_oran, oy_sayisi, yorum_kapsami FROM oyunlar WHERE app_id = 1222670")),
    ("50 satır sınırı", lambda: sql_araci.sql_calistir("SELECT ad FROM oyunlar WHERE oy_sayisi >= 100000 ORDER BY oy_sayisi DESC")[:300]),
    ("metin kapalı", lambda: sql_araci.sql_calistir("SELECT metin FROM yorumlar WHERE app_id = 239140 LIMIT 1")),
    ("salt okuma", lambda: sql_araci.sql_calistir("DELETE FROM oyunlar WHERE app_id = -1")),
    ("SELECT değil", lambda: sql_araci.sql_calistir("PRAGMA foreign_keys = ON")),
    ("zaman aşımı", lambda: sql_araci.sql_calistir("SELECT AVG(oynama_saat) FROM yorumlar")),
]
for baslik, f in testler:
    t = time.perf_counter()
    cikti = f()
    print(f"########## {baslik} ({time.perf_counter() - t:.2f} sn)\n{cikti}\n")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
DB: /home/kaan/steam-oyun-asistani/data/processed/steam.db | var mı: True
oyun_adlari: 64948 satır

########## oyun_bul (0.18 sn)
'sims 4' için adı en çok benzeyen 5 oyun (puan = ad benzerliği 0-100, kesin eşleşme anlamına gelmez):
app_id,ad,oy_sayisi,puan
1222670,The Sims™ 4,153114,90
2209440,sim4D ATC,110,80
3118260,SIM - Simple Idle Miner,8,80
3103230,Sim Solver,0,80
1313490,Slimes,37,73
Kural: Sorulan oyunu ad ve oy sayısına bakarak seç. Adaylardan hiçbiri sorulan oyun değilse (farklı bir oyun, Playtest, araç veya kart oyunu) oyunun veri setinde olmadığını söyle. Seçtiğin oyunu sonraki SQL sorgularında app_id ile kullan.

########## normal sorgu (0.00 sn)
1 satır, 0.00 sn
ad,olumlu_oran,oy_sayisi,yorum_kapsami
The Sims™ 4,0.874,153114,0.06


########## 50 satır sınırı (3.13 sn)
DİKKAT: Sonuç 50 satırda kesildi. Sorgunun toplam sonucu 82 satır, sadece ilk 50 tanesi gösteriliyor. Sayı lazımsa satı

## Adım 1a · SQL aracı src/'a taşındı
- Dosyalar: `src/__init__.py`, `src/ayarlar.py` (yollar + sabitler, proje köküne göre), `src/sql_araci.py` (04'ün son hâli).
- 04'e göre farklar: `sql_calistir` + zaman aşımı tek fonksiyon; her sorgu kendi salt okunur bağlantısını açıp kapatıyor (API'de eşzamanlı istek için); SELECT olmayan sorguya anlaşılır HATA; `oyun_bul` çıktısı CSV.
- Kontrol (04 ile aynı): Sims 4 %87,4 / kapsam 0,06 · 100k+ oylu oyun 50'de kesildi, toplam 82 · metin sütunu reddedildi · yazma reddedildi · PRAGMA → HATA.
- Zaman aşımı: `AVG(oynama_saat)` bu sefer 9,99 sn'de bitti (önbellek ılık), sınırın hemen altında. Durdurma 1b'de 2 sn ile doğrulanacak.
- Notebook'ta `sys.path.insert(0, "..")` + `%autoreload 2`: src'deki değişiklik kernel'i yeniden başlatmadan görülür.

In [9]:
import json
from src import sql_araci, araclar

sql_araci.ZAMAN_ASIMI_SN = 2
t = time.perf_counter()
print(sql_araci.sql_calistir("SELECT AVG(oynama_saat) FROM yorumlar"), f"({time.perf_counter() - t:.1f} sn)\n")
sql_araci.ZAMAN_ASIMI_SN = 10

print(f"Isınma: {araclar.isinma():.1f} sn")
print("Araçlar:", [a["function"]["name"] for a in araclar.ARAC_TANIMLARI])
print(f"Tanımların boyutu ≈ {len(json.dumps(araclar.ARAC_TANIMLARI, ensure_ascii=False)) / 3.4:,.0f} token\n")

testler = [
    ("konu, DL gece",       "yorum_ara", '{"mod": "konu", "soru": "Gece oynamak korkutucu mu?", "app_id": 239140}', 700),
    ("ozetle, Sims 4",      "yorum_ara", '{"mod": "ozetle", "app_id": 1222670}', 700),
    ("hepsi, CP bug",       "yorum_ara", '{"mod": "hepsi", "soru": "Oyunda bug ve hata çok mu?", "app_id": 1091500}', 900),
    ("yorum yok, Starfield","yorum_ara", '{"mod": "ozetle", "app_id": 1716740}', None),
    ("oyun_ara, uzay",      "oyun_ara",  '{"istek": "uzayda geçen, gezegen keşfedilen açık dünya rol yapma oyunu"}', None),
    ("SQL, araclar'dan",    "sql_calistir", '{"sorgu": "SELECT COUNT(*) AS n FROM oyunlar WHERE oy_sayisi >= 100000"}', None),
    ("yanlış parametre",    "sql_calistir", '{"query": "SELECT 1"}', None),
    ("bilinmeyen araç",     "web_ara",   '{"q": "sims"}', None),
]
for baslik, ad, arg, kes in testler:
    t = time.perf_counter()
    cikti = araclar.arac_cagir(ad, arg)
    print(f"########## {baslik} ({time.perf_counter() - t:.2f} sn, {len(cikti):,} karakter)")
    print(cikti[:kes] if kes else cikti, "\n")

HATA: Sorgu 2 saniyeyi geçtiği için durduruldu. Daha dar bir sorgu yaz: app_id veya indeksli sütunlarla (dil, tarih) filtrele, bütün yorumlar tablosunu tarayan sorgulardan kaçın. (2.0 sn)

Isınma: 0.1 sn
Araçlar: ['sql_calistir', 'oyun_bul', 'yorum_ara', 'oyun_ara']
Tanımların boyutu ≈ 2,654 token

########## konu, DL gece (0.29 sn, 2,218 karakter)
KONU ARAMASI: 5 yorum, en ilgiliden aşağı. Mesafe 0.45 üstü = zayıf eşleşme, kanıt sayma.

[app_id=239140 | olumlu | dil=zh | 2022-10-25 | faydalı=837 | mesafe=0.329]
晚上比早上安全得多，晚上是比较适合新手探索。而且没有任何恐怖元素，胆小的人可以尝试。夜魔名字听起来可怕，其实夜魔挺温和的而且经验很多，不要被其他评论影响了。这游戏也很适合儿童游玩

[app_id=239140 | olumlu | dil=en | 2015-01-31 | faydalı=684 | mesafe=0.332]
"The Walking Dead" by day, "I Am Legend" by night;I can't tell you how beautiful this game looks without you just seeing it for yourself,  but I will say this.This is the first game, in a very long time, which has legitamately inspired a sense of fear in me.At night time, this game becomes a heart pounding nightma

## Adım 1b · RAG aracı src/'a taşındı, araçlar birleşti
- `src/rag_araci.py`: 06'nın son hâli (id eşlemesi, özet = en yakın yarı, oyun_ara w = 0,03) + `isinma()`.
- `src/araclar.py`: `ARAC_TANIMLARI` (sql_calistir, oyun_bul, yorum_ara, oyun_ara) + tek `arac_cagir` (04 + 06 hata yönetimi birleşik). Agent sadece bunu görür.
- Tanımların boyutu ≈ 2.654 token → her LLM çağrısında sabit maliyet.
- Kontrol (06 ile aynı): DL gece 0,329 · Sims 175/25 · CP 37 aday (evet 24 / hayır 11 / ilgisiz 2) · Starfield yorum yok · uzay: Starfield, Outer Wilds · SQL 82 · yanlış parametre ve bilinmeyen araç → HATA.
- Zaman aşımı 2 sn ile doğrulandı.
- Not: RAG yüklendikten sonra SQL COUNT 0,08 → 3,5 sn (RAM baskısı / soğuk önbellek şüphesi) → 08'de ölç.
- Not: Sims özetinde korsan DLC + seks modu tavsiyesi var → agent kural 5 ile süzmeli (Adım 5'te test).

In [10]:
import os, requests
from src import llm

kurulu_ollama = [m["name"] for m in requests.get(f"{sql_araci.ayarlar.OLLAMA_URL}/api/tags").json()["models"]]
print("Ollama'da kurulu modeller:", kurulu_ollama, "\n")

mesajlar = [
    {"role": "system", "content": "Sen Steam oyunları hakkında Türkçe cevap veren bir asistansın. Cevap için araçları kullan."},
    {"role": "user", "content": "The Sims 4'ün Steam puanı kaç?"},
]

for model, saglayici in llm.MODELLER.items():
    anahtar = llm.SAGLAYICILAR[saglayici]["anahtar"]
    if anahtar and not os.getenv(anahtar):
        print(f"########## {model}: ATLANDI ({anahtar} .env'de yok)\n")
        continue
    if saglayici == "ollama" and model not in kurulu_ollama:
        print(f"########## {model}: ATLANDI (Ollama'da kurulu değil)\n")
        continue
    t = time.perf_counter()
    try:
        cevap = llm.sohbet(model, mesajlar, araclar=araclar.ARAC_TANIMLARI)
    except Exception as e:
        print(f"########## {model}: HATA {type(e).__name__}: {e}\n")
        continue
    sure = time.perf_counter() - t
    m = cevap.choices[0].message
    print(f"########## {model} ({sure:.1f} sn) | bitiş: {cevap.choices[0].finish_reason} | "
          f"girdi {cevap.usage.prompt_tokens} / çıktı {cevap.usage.completion_tokens} token")
    for c in m.tool_calls or []:
        print(f"  ARAÇ İSTEĞİ: {c.function.name}({c.function.arguments})  id={c.id}")
    print(f"  METİN: {m.content!r}\n")

Ollama'da kurulu modeller: ['qwen3-embedding:0.6b', 'qwen3-embedding:4b', 'bge-m3:latest'] 

########## gpt-4.1-mini (0.7 sn) | bitiş: tool_calls | girdi 2810 / çıktı 20 token
  ARAÇ İSTEĞİ: oyun_bul({"isim":"The Sims 4"})  id=call_Rv6novKGKT5PB8KxAsqpjDdQ
  METİN: None

########## gemini-2.5-flash: ATLANDI (GEMINI_API_KEY .env'de yok)

########## deepseek-chat: ATLANDI (DEEPSEEK_API_KEY .env'de yok)

########## qwen2.5:7b: ATLANDI (Ollama'da kurulu değil)

########## qwen2.5:14b: ATLANDI (Ollama'da kurulu değil)



## Adım 2 · LLM bağlantısı (src/llm.py)
- Bütün sağlayıcılar OpenAI uyumlu API kullanıyor → değişen tek şey `base_url` + anahtar. `MODELLER` sözlüğünde model adı → sağlayıcı; `sohbet(model, mesajlar, araclar)` projenin LLM'e giden tek kapısı (temperature 0).
- Tool calling'in özü: modele araç tanımları (JSON şema) verilir; model cevap yerine `tool_calls` döndürebilir ("şu aracı şu argümanla çalıştır"). Aracı modelin kendisi çalıştırmaz, biz çalıştırırız.
- Test "The Sims 4'ün Steam puanı kaç?" → gpt-4.1-mini: `finish_reason = tool_calls`, `oyun_bul({"isim": "The Sims 4"})`, metin boş. 0,7 sn, girdi 2.810 token (≈2.650'si araç tanımları = her turun sabit maliyeti).
- Geliştirme modeli: gpt-4.1-mini. 09 için eksik: GEMINI_API_KEY, DEEPSEEK_API_KEY, `ollama pull qwen2.5:7b / 14b`.

In [11]:
from src.talimat import SISTEM_TALIMATI

print(f"Talimat: {len(SISTEM_TALIMATI):,} karakter ≈ {len(SISTEM_TALIMATI) / 3.4:,.0f} token\n")

sorular = [
    ("puan",            "Elden Ring'in Steam puanı kaç?"),
    ("sayım",           "100 binden fazla oy alan kaç oyun var?"),
    ("yorum içeriği",   "Dying Light'ta oyun çöküyor mu?"),
    ("tarif",           "Arkadaşlarımla oynayabileceğim rahatlatıcı bir oyun öner"),
    ("kesin kriter",    "Türkçe dil desteği olan en yüksek puanlı korku oyunları hangileri?"),
    ("selam",           "Merhaba, neler yapabiliyorsun?"),
    ("kapsam dışı",     "Python'da bir listeyi nasıl sıralarım?"),
    ("oyun yok",        "Witcher 3 nasıl bir oyun?"),
]

for tip, soru in sorular:
    mesajlar = [{"role": "system", "content": SISTEM_TALIMATI}, {"role": "user", "content": soru}]
    t = time.perf_counter()
    cevap = llm.sohbet("gpt-4.1-mini", mesajlar, araclar=araclar.ARAC_TANIMLARI)
    m = cevap.choices[0].message
    print(f"########## [{tip}] {soru} ({time.perf_counter() - t:.1f} sn, girdi {cevap.usage.prompt_tokens} token)")
    for c in m.tool_calls or []:
        print(f"  ARAÇ: {c.function.name}({c.function.arguments})")
    if m.content:
        print(f"  METİN: {m.content}")
    print()

Talimat: 2,541 karakter ≈ 747 token

########## [puan] Elden Ring'in Steam puanı kaç? (0.7 sn, girdi 3597 token)
  ARAÇ: oyun_bul({"isim":"Elden Ring"})

########## [sayım] 100 binden fazla oy alan kaç oyun var? (0.7 sn, girdi 3598 token)
  ARAÇ: sql_calistir({"sorgu":"SELECT COUNT(*) AS oyun_sayisi FROM oyunlar WHERE oy_sayisi > 100000;"})

########## [yorum içeriği] Dying Light'ta oyun çöküyor mu? (0.7 sn, girdi 3599 token)
  ARAÇ: oyun_bul({"isim": "Dying Light"})
  ARAÇ: oyun_bul({"isim": "Dying Light 2"})

########## [tarif] Arkadaşlarımla oynayabileceğim rahatlatıcı bir oyun öner (0.8 sn, girdi 3602 token)
  ARAÇ: oyun_ara({"istek":"rahatlatıcı oyun co-op"})

########## [kesin kriter] Türkçe dil desteği olan en yüksek puanlı korku oyunları hangileri? (1.2 sn, girdi 3607 token)
  ARAÇ: sql_calistir({"sorgu":"SELECT o.ad, o.olumlu_oran, o.oy_sayisi, o.fiyat FROM oyunlar o JOIN oyun_etiket e ON o.app_id = e.app_id JOIN oyun_dil d ON o.app_id = d.app_id WHERE e.etiket = 'Horror' AND 

## Adım 3 · Sistem talimatı (src/talimat.py)
- Ayrım: araca özel kurallar araç açıklamalarında (04: 10 kural, 06: 9 kural); sistem talimatında araçlar arası iş akışı: araç seçimi, doğruluk (7 kural), hata davranışı, cevap biçimi.
- Boyut 747 token → her tur ≈ 3.600 token girdi (araç tanımları 2.650 + talimat 750).
- İlk tur yönlendirme 8/8 doğru: puan / çökme / Witcher → oyun_bul · sayım → doğrudan COUNT · tarif → oyun_ara · Türkçe + korku → SQL (etiket Horror + dil Turkish + oy ≥ 100) · selam → araçsız tanıtım · Python → kibar ret.
- Paralel tool call: model bir turda 2 araç istedi (Dying Light + Dying Light 2) → döngü bir turdaki bütün araç isteklerini çalıştırmalı.
- İzlenecek: korku listesinde `yetiskin = 0` filtresi yok (kural 6) → Adım 5'te cevapta kontrol.

In [12]:
MODEL = "gpt-4.1-mini"
MAKS_TUR = 8

soru = "The Sims 4'ü oyuncular seviyor mu?"
mesajlar = [{"role": "system", "content": SISTEM_TALIMATI},
            {"role": "user", "content": soru}]

t0 = time.perf_counter()
toplam_girdi = toplam_cikti = 0

for tur in range(1, MAKS_TUR + 1):
    cevap = llm.sohbet(MODEL, mesajlar, araclar=araclar.ARAC_TANIMLARI)
    m = cevap.choices[0].message
    toplam_girdi += cevap.usage.prompt_tokens
    toplam_cikti += cevap.usage.completion_tokens

    if not m.tool_calls:
        print(f"===== TUR {tur}: CEVAP =====")
        break

    mesajlar.append(m.model_dump(exclude_none=True))
    for c in m.tool_calls:
        t = time.perf_counter()
        sonuc = araclar.arac_cagir(c.function.name, c.function.arguments)
        mesajlar.append({"role": "tool", "tool_call_id": c.id, "content": sonuc})
        print(f"===== TUR {tur}: {c.function.name}({c.function.arguments}) → {len(sonuc):,} karakter, {time.perf_counter() - t:.2f} sn")
        print(sonuc[:300], "\n")
else:
    print(f"===== {MAKS_TUR} TUR DOLDU, araçsız son cevap isteniyor =====")
    cevap = llm.istemci(MODEL).chat.completions.create(
        model=MODEL, messages=mesajlar, tools=araclar.ARAC_TANIMLARI, tool_choice="none", temperature=0)
    m = cevap.choices[0].message

print(m.content)
print(f"\nToplam: {time.perf_counter() - t0:.1f} sn | {tur} tur | girdi {toplam_girdi:,} / çıktı {toplam_cikti:,} token | "
      f"mesaj sayısı {len(mesajlar)}")

===== TUR 1: oyun_bul({"isim":"The Sims 4"}) → 552 karakter, 0.24 sn
'The Sims 4' için adı en çok benzeyen 5 oyun (puan = ad benzerliği 0-100, kesin eşleşme anlamına gelmez):
app_id,ad,oy_sayisi,puan
1222670,The Sims™ 4,153114,100
2263510,The Simulacrum,112,77
1502540,The Sisters - Party of the Year,10,77
1569560,The Simplest Game,10,77
2541150,The Simplest Game in t 

===== TUR 2: sql_calistir({"sorgu":"SELECT olumlu_oran, oy_sayisi, yorum_kapsami FROM oyunlar WHERE app_id = 1222670"}) → 71 karakter, 0.00 sn
1 satır, 0.00 sn
olumlu_oran,oy_sayisi,yorum_kapsami
0.874,153114,0.06
 

===== TUR 3: yorum_ara({"mod":"ozetle","app_id":1222670}) → 3,364 karakter, 5.36 sn
ÖZET: örneklemde 200 yorum var (olumlu 175, olumsuz 25). Bu oran genel puan değildir, genel puanı SQL'den al. Aşağıdakiler oyunu değerlendiren en faydalı yorumlar.

=== OLUMLU ===
[app_id=1222670 | olumlu | dil=en | 2024-06-30 | faydalı=1488]
best game ever. if you want all DLCs just search up Anadi 

===== TUR 4: CEVAP =====

## Adım 4 · Elle tool-calling döngüsü
- Döngü: LLM çağır → tool_calls yoksa DUR (cevap) → varsa assistant mesajını geçmişe ekle, her isteği çalıştır, sonucu {"role": "tool", "tool_call_id": ...} olarak ekle → tekrar. `mesajlar` agent'ın tek hafızası; her tur tamamı modele gider.
- Emniyet: MAKS_TUR = 8; dolarsa `tool_choice="none"` ile araçsız son cevap.
- Test "The Sims 4'ü oyuncular seviyor mu?": oyun_bul → SQL (0,874 / 153.114 / 0,06) → ozetle → cevap. 4 tur, 9,2 sn (ozetle soğuk 5,4 sn), girdi 16.889 / çıktı 231 token (≈ 0,7 sent).
- Cevap kalitesi: puan SQL'den ✓, çelişki açıklandı (DLC fiyatı) ✓, kapsam uyarısı ✓, korsan/seks modu aktarılmadı ✓ (kural 5). Küçük kayma: "yorumların %6'sı" (doğrusu: oyların %6'sının yorumu).
- Model SQL + özetle'yi art arda istedi (paralel değil) → girdi tokenı tur başına birikiyor.

In [13]:
import pandas as pd
from src import agent

testler = [
    ("oyun yok",        "Witcher 3 nasıl bir oyun?"),
    ("belirsiz ad",     "Spider-Man'in puanı kaç?"),
    ("yorum yok",       "Starfield hakkında oyuncular ne diyor?"),
    ("düzeldi mi",      "Cyberpunk 2077'deki bug'lar zamanla düzeldi mi?"),
    ("öneri",           "Bana iyi bir korku oyunu öner"),
    ("çocuk",           "9 yaşındaki çocuğumla oynayabileceğim bir oyun öner"),
    ("kesin kriter",    "Türkçe dil desteği olan en yüksek puanlı korku oyunları hangileri?"),
]

sonuclar = []
for tip, soru in testler:
    s = agent.calistir(soru)
    s["tip"] = tip
    sonuclar.append(s)
    print(f"########## [{tip}] {soru}")
    print(f"{s['tur']} tur | {s['sure']} sn | girdi {s['girdi_token']:,} token | tur doldu: {s['tur_doldu']}")
    for a in s["araclar"]:
        print("  →", a)
    print(f"\n{s['cevap']}\n")

ozet = pd.DataFrame(sonuclar)[["tip", "tur", "sure", "girdi_token", "cikti_token"]]
ozet["arac_sayisi"] = [len(s["araclar"]) for s in sonuclar]
ozet["hata"] = [sum(a.get("hata", False) for a in s["iz"]) for s in sonuclar]
print(ozet.to_string(index=False))
print(f"\nToplam girdi {ozet['girdi_token'].sum():,} token ≈ ${ozet['girdi_token'].sum() * 0.40 / 1e6 + ozet['cikti_token'].sum() * 1.60 / 1e6:.3f}")

agent.iz_kaydet(sonuclar, "../reports/07_test_izleri.jsonl")

########## [oyun yok] Witcher 3 nasıl bir oyun?
2 tur | 1.7 sn | girdi 7,417 token | tur doldu: False
  → oyun_bul({"isim":"Witcher 3"})

Veri setinde "Witcher 3" adlı oyun bulunmamaktadır. Benzer adlarda "The Witcher 3 REDkit Playtest" ve "GWENT: The Witcher Card Game" gibi oyunlar var ama bunlar asıl Witcher 3 oyunu değil. Başka bir oyun hakkında bilgi isterseniz yardımcı olabilirim.

########## [belirsiz ad] Spider-Man'in puanı kaç?
3 tur | 2.2 sn | girdi 11,288 token | tur doldu: False
  → oyun_bul({"isim":"Spider-Man"})
  → sql_calistir({"sorgu":"SELECT ad, olumlu_oran, oy_sayisi FROM oyunlar WHERE app_id = 1817070"})

Marvel’s Spider-Man Remastered oyununun Steam'deki puanı %95,9'dur. Bu bilgi 101.951 oy temel alınarak verilmiştir.

########## [yorum yok] Starfield hakkında oyuncular ne diyor?
3 tur | 2.2 sn | girdi 11,251 token | tur doldu: False
  → oyun_bul({"isim":"Starfield"})
  → yorum_ara({"mod":"ozetle","app_id":1716740})

Starfield için elimizde oyuncu yorum verisi yok. 

## Adım 5 · src/agent.py + 7 test sorusu (talimat v1)
- `agent.calistir(soru)` → cevap, tur, süre, token, araç sırası, iz, gecmis (sohbet devamı). İzler reports/07_test_izleri.jsonl.
- Toplam: 7 soru, 2–3 tur, 1,7–5,1 sn, 68.396 girdi token ≈ $0,029 (soru başı ~0,4 sent).
- ✅ Witcher 3 yok · ✅ Cyberpunk "çıkışta bug'lı, sonra azaldı" (hepsi + yıllar) · ✅ Türkçe korku (SQL).
- ⚠️ Spider-Man: Remastered seçildi, Miles Morales anılmadı.
- ❌ Starfield: "%81,7" — SQL çağrısı yok → uydurma şüphesi (doğrulanacak).
- ❌ Korku öner: saf olumlu_oran sıralaması → 108 oylu %100 bilinmeyen oyunlar. SQL hatası (ambiguous app_id) agent tarafından düzeltildi.
- ❌ Çocuk: uygunluk filtresi yok, korku etiketli oyunlar önerildi; fiyat birimi "TL" (veri $).
- Ders: tanımlanmamış kavramları ("iyi oyun", "çocuğa uygun") model kendi doldurur → talimatta açık tanım gerekli.

In [14]:
s = sql_araci.sql_calistir

print("##### 1) Starfield gerçek değerler")
print(s("SELECT ad, olumlu_oran, oy_sayisi, yorum_var FROM oyunlar WHERE app_id = 1716740"))
starfield = next(x for x in sonuclar if x["tip"] == "yorum yok")
arac_ciktilari = [m["content"] for m in starfield["gecmis"] if m["role"] == "tool"]
print("Araç çıktılarında '81' geçiyor mu:", any("81" in c for c in arac_ciktilari))
for c in arac_ciktilari:
    print("---", c)
print()

print("##### 2) Çocuk kuralı için etiketler")
print(s("""SELECT etiket, COUNT(*) AS oyun FROM oyun_etiket
WHERE etiket IN ('Family Friendly','Cute','Colorful','Education','Horror','Psychological Horror','Survival Horror',
                 'Gore','Violent','Blood','Nudity','Sexual Content','Mature','NSFW')
GROUP BY etiket ORDER BY oyun DESC"""))

print("##### 3a) Korku: oy >= 1000, orana göre")
print(s("""SELECT o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o JOIN oyun_etiket e ON e.app_id = o.app_id
WHERE e.etiket = 'Horror' AND o.yetiskin = 0 AND o.oy_sayisi >= 1000 ORDER BY o.olumlu_oran DESC LIMIT 10"""))
print("##### 3b) Korku: oran >= 0,85, oy sayısına göre")
print(s("""SELECT o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o JOIN oyun_etiket e ON e.app_id = o.app_id
WHERE e.etiket = 'Horror' AND o.yetiskin = 0 AND o.olumlu_oran >= 0.85 ORDER BY o.oy_sayisi DESC LIMIT 10"""))
print("##### 3c) Resident Evil Village nerede?")
print(s("""SELECT o.app_id, o.ad, o.olumlu_oran, o.oy_sayisi, o.yetiskin FROM oyunlar o
WHERE o.ad LIKE '%Resident Evil Village%'"""))

print("##### 4) Fiyat birimi (bilinen fiyatlar: Elden Ring 59,99 $, BG3 59,99 $)")
print(s("SELECT ad, fiyat, normal_fiyat FROM oyunlar WHERE app_id IN (1245620, 1086940)"))

##### 1) Starfield gerçek değerler
1 satır, 0.00 sn
ad,olumlu_oran,oy_sayisi,yorum_var
Starfield,0.57,160947,0

Araç çıktılarında '81' geçiyor mu: False
--- 'Starfield' için adı en çok benzeyen 5 oyun (puan = ad benzerliği 0-100, kesin eşleşme anlamına gelmez):
app_id,ad,oy_sayisi,puan
1716740,Starfield,160947,100
2588820,Asteroids Star Fields,0,90
1523900,Warfield,13,82
2540820,Starfend,11,82
2774120,Vstar,0,80
Kural: Sorulan oyunu ad ve oy sayısına bakarak seç. Adaylardan hiçbiri sorulan oyun değilse (farklı bir oyun, Playtest, araç veya kart oyunu) oyunun veri setinde olmadığını söyle. Seçtiğin oyunu sonraki SQL sorgularında app_id ile kullan.
--- Bu oyun için yorum verisi yok. Sadece SQL'deki sayıları kullan ve bunu kullanıcıya söyle.

##### 2) Çocuk kuralı için etiketler
14 satır, 0.03 sn
etiket,oyun
Colorful,10320
Cute,8560
Horror,5751
Family Friendly,5604
Psychological Horror,3642
Violent,2382
Survival Horror,2377
Gore,2037
Sexual Content,1751
Education,1517
Nudity,1377
Mature,7

## Adım 5b · Hataların kaynağı (SQL kontrolü)
- Starfield: gerçek olumlu_oran 0,57 (160.947 oy); agent %81,7 dedi; araç çıktılarında "81" yok → UYDURMA kesin. Model oy sayısını oyun_bul'dan aldı, oranı kendi bilgisinden ekledi.
- Etiketler (oyun sayısı): Family Friendly 5.604, Cute 8.560 · hariç adaylar: Horror 5.751, Psychological Horror 3.642, Survival Horror 2.377, Violent 2.382, Gore 2.037, Blood 280, Sexual Content 1.751, Nudity 1.377, Mature 779, NSFW 553.
- Korku sıralaması: oy ≥ 1000 + orana göre → az bilinen yüksek puanlılar (A Castle Full of Cats %99,4); oran ≥ 0,85 + oy sayısına göre → popülerler (Phasmophobia, Lethal Company, RE4). RE Village %95,6 / 113.043.
- Fiyat ABD doları (Elden Ring 59,99). `fiyat` indirimli olabilir (BG3 47,99 / normal 59,99).

In [1]:
%load_ext autoreload
%autoreload 2
import sys, time, json
sys.path.insert(0, "..")

import pandas as pd
from src import agent, araclar

print(araclar.arac_cagir("oyun_bul", '{"isim": "starfield"}'), "\n")

testler_v2 = [
    ("yorum yok",        "Starfield hakkında oyuncular ne diyor?"),
    ("belirsiz ad",      "Spider-Man'in puanı kaç?"),
    ("öneri",            "Bana iyi bir korku oyunu öner"),
    ("çocuk",            "9 yaşındaki çocuğumla oynayabileceğim bir oyun öner"),
    ("en yüksek puanlı", "Türkçe dil desteği olan en yüksek puanlı korku oyunları hangileri?"),
]

sonuclar_v2 = []
for tip, soru in testler_v2:
    s = agent.calistir(soru)
    s["tip"] = tip
    s["talimat"] = "v2"
    sonuclar_v2.append(s)
    print(f"########## [{tip}] {soru}")
    print(f"{s['tur']} tur | {s['sure']} sn | girdi {s['girdi_token']:,} token")
    for a in s["araclar"]:
        print("  →", a)
    print(f"\n{s['cevap']}\n")

ozet = pd.DataFrame(sonuclar_v2)[["tip", "tur", "sure", "girdi_token"]]
ozet["hata"] = [sum(a.get("hata", False) for a in s["iz"]) for s in sonuclar_v2]
print(ozet.to_string(index=False))
agent.iz_kaydet(sonuclar_v2, "../reports/07_test_izleri.jsonl")

'starfield' için adı en çok benzeyen 5 oyun (benzerlik = ad benzerliği 0-100, kesin eşleşme anlamına gelmez; olumlu_oran = Steam oylarından genel puan, 0-1, boşsa oy yok):
app_id,ad,oy_sayisi,olumlu_oran,benzerlik
1716740,Starfield,160947,0.57,100
2588820,Asteroids Star Fields,0,,90
1523900,Warfield,13,0.308,82
2540820,Starfend,11,1.0,82
2774120,Vstar,0,,80
Kural: Sorulan oyunu ad ve oy sayısına bakarak seç. Adaylardan hiçbiri sorulan oyun değilse (farklı bir oyun, Playtest, araç veya kart oyunu) oyunun veri setinde olmadığını söyle. Seçtiğin oyunu sonraki SQL sorgularında app_id ile kullan. 

########## [yorum yok] Starfield hakkında oyuncular ne diyor?
3 tur | 2.3 sn | girdi 12,296 token
  → oyun_bul({"isim":"Starfield"})
  → yorum_ara({"mod":"ozetle","app_id":1716740})

Starfield oyunu Steam'de 160.947 oy almış ve genel olumlu oy oranı %57'dir. Ancak elimizde oyuncu yorumları bulunmamaktadır, bu yüzden oyuncuların detaylı görüşlerini özetleyemiyorum. Bu bilgi Steam oyun verilerine d

## Adım 5c · Talimat v2 + oyun_bul'a olumlu_oran
- Değişiklikler: kural 1 (her sayı araç çıktısında geçmeli), kural 3 (diğer adayı da an), kural 6 (sıralama: öner → oran ≥ 0,85 + popülerlik; "en yüksek puanlı" → oy ≥ 1000 + oran), kural 7 (çocuk: Family Friendly + 10 etiket NOT EXISTS), kural 8 (fiyat $). oyun_bul çıktısına olumlu_oran; "puan" → "benzerlik".
- Düzelenler: Starfield %57 ✅ (yapısal düzeltme: oran oyun_bul'da görünüyor; talimat yüklenmeden de düzelmişti) · Spider-Man iki adayı da verip sordu ✅ · çocuk SQL'i etiket filtreli, fiyat $ ✅.
- Düzelmeyen: sıralama. Sebep: SQL araç açıklamasındaki (04) kural 3 "oy_sayisi >= 100" ile sistem talimatı çelişiyor; model SQL yazarken araç açıklamasına uyuyor.
- Ders: kurallar tek kaynaktan, tutarlı olmalı. Yapısal düzeltme (veriyi araç çıktısına koymak) talimattan daha güvenilir.
- Not: ilk denemede talimat.py güncellenmemişti; girdi token sayısı (7.464 = v1) bunu ele verdi.

In [1]:
%load_ext autoreload
%autoreload 2
import sys, time, json
sys.path.insert(0, "..")

import pandas as pd
from src import agent, talimat, sql_araci

print("talimat:", len(talimat.SISTEM_TALIMATI), "| SQL kural 3 yeni mi:", "YAPMA" in sql_araci.SQL_ARAC_ACIKLAMASI, "\n")

testler_v3 = [
    ("öneri",            "Bana iyi bir korku oyunu öner"),
    ("çocuk",            "9 yaşındaki çocuğumla oynayabileceğim bir oyun öner"),
    ("en yüksek puanlı", "Türkçe dil desteği olan en yüksek puanlı korku oyunları hangileri?"),
]

sonuclar_v3 = []
for tip, soru in testler_v3:
    s = agent.calistir(soru)
    s["tip"] = tip
    s["talimat"] = "v3"
    sonuclar_v3.append(s)
    print(f"########## [{tip}] {soru}")
    print(f"{s['tur']} tur | {s['sure']} sn | girdi {s['girdi_token']:,} token")
    for a in s["araclar"]:
        print("  →", a)
    print(f"\n{s['cevap']}\n")

agent.iz_kaydet(sonuclar_v3, "../reports/07_test_izleri.jsonl")

talimat: 3585 | SQL kural 3 yeni mi: True 

########## [öneri] Bana iyi bir korku oyunu öner
2 tur | 2.7 sn | girdi 8,415 token
  → sql_calistir({"sorgu":"SELECT o.app_id, o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o JOIN oyun_etiket e ON o.app_id = e.app_id WHERE e.etiket = 'Horror' AND o.olumlu_oran >= 0.85 AND o.oy_sayisi >= 100 AND o.yetiskin = 0 AND NOT EXISTS (SELECT 1 FROM oyun_etiket e2 WHERE e2.app_id = o.app_id AND e2.etiket IN ('Psychological Horror', 'Survival Horror', 'Gore', 'Violent', 'Blood', 'Sexual Content', 'Nudity', 'Mature', 'NSFW')) ORDER BY o.oy_sayisi DESC LIMIT 5;"})

İyi bir korku oyunu olarak "7 Days to Die" öneririm. Steam'de %88 olumlu oy oranı ve 342.380 oy sayısıyla oldukça popüler ve beğenilen bir oyun. Ancak bu oyun aynı zamanda hayatta kalma ve açık dünya unsurları da içeriyor.

Not: "Deep Rock Galactic", "Stray", "Inscryption" ve "Grounded" da yüksek puanlı ve popüler oyunlar ama korku türü olarak daha az uygun olabilir.

Steam oyunlarına göre ve 1

## Adım 5d · Kural çelişkisi kaldırıldı (v3)
- SQL araç açıklaması kural 3 talimatla aynı yapıldı, kalıp SQL parçasıyla: öner → `olumlu_oran >= 0.85 AND oy_sayisi >= 100 ORDER BY oy_sayisi DESC`; "en yüksek puanlı" → `oy_sayisi >= 1000 ORDER BY olumlu_oran DESC` + eşiği söyle.
- Sonuç: sıralama 3/3 doğru. Çocuk ✅ (Hogwarts Legacy, Sims 4, TABS, Stumble Guys, Farming Sim 22 + ebeveyn uyarısı). En yüksek puanlı ✅ (≥ 1000 oy, eşik söylendi; JOIN + LIKE '%Horror%' → Half-Life: Alyx iki satır).
- Yeni hata: "korku öner"de çocuk hariç listesi uygulandı → Phasmophobia/RE4 elendi. Kuralın fazla genelleştirilmesi.
- Ders: her talimat değişikliği başka soruyu bozabilir → değişiklikten sonra bütün test seti tekrar (08).
- Düzeltme (v4, Adım 6'da test): kural 7 sadece çocuk/aile; SQL kural 11 etiket filtresi EXISTS.

In [2]:
%pip install langgraph

  Using cached langgraph-1.2.19-py3-none-any.whl.metadata (4.9 kB)
  Using cached langchain_core-1.7.3-py3-none-any.whl.metadata (4.8 kB)
  Using cached langgraph_checkpoint-4.3.1-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.1-py3-none-any.whl.metadata (6.9 kB)
  Using cached langgraph_sdk-0.4.11-py3-none-any.whl.metadata (5.1 kB)
  Using cached xxhash-4.0.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached jsonpatch-1.38-py3-none-any.whl.metadata (3.7 kB)
  Using cached langchain_protocol-0.0.20-py3-none-any.whl.metadata (2.4 kB)
  Using cached langsmith-0.14.16-py3-none-any.whl.metadata (39 kB)
  Using cached uuid_utils-1.2.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
  Using cached ormsgpack-1.13.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (3.2 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests_t

In [1]:
%load_ext autoreload
%autoreload 2
import sys, time, json
sys.path.insert(0, "..")
import pandas as pd
from src import agent, agent_graf, talimat, sql_araci

print("talimat:", len(talimat.SISTEM_TALIMATI), "| SQL kural 11:", "EXISTS ile yaz" in sql_araci.SQL_ARAC_ACIKLAMASI)
print(agent_graf.GRAF.get_graph().draw_mermaid())

testler_v4 = [
    ("öneri",            "Bana iyi bir korku oyunu öner"),
    ("en yüksek puanlı", "Türkçe dil desteği olan en yüksek puanlı korku oyunları hangileri?"),
    ("çelişki",          "The Sims 4'ü oyuncular seviyor mu?"),
]

satirlar, sonuclar_v4 = [], []
for tip, soru in testler_v4:
    e = agent.calistir(soru)
    g = agent_graf.calistir(soru, oturum=f"test-{tip}")
    for ad, s in [("elle", e), ("langgraph", g)]:
        s.update(tip=tip, talimat="v4", surum=ad)
        sonuclar_v4.append(s)
    satirlar.append({"tip": tip, "elle_tur": e["tur"], "graf_tur": g["tur"], "elle_sn": e["sure"], "graf_sn": g["sure"],
                     "elle_token": e["girdi_token"], "graf_token": g["girdi_token"],
                     "ayni_araclar": [a.split("(")[0] for a in e["araclar"]] == [a.split("(")[0] for a in g["araclar"]]})
    print(f"\n########## [{tip}] {soru}")
    for ad, s in [("ELLE", e), ("LANGGRAPH", g)]:
        print(f"--- {ad}: {' → '.join(s['araclar'])}\n{s['cevap']}\n")

print(pd.DataFrame(satirlar).to_string(index=False))

print("\n########## HAFIZA: aynı oturumda ikinci soru")
g1 = agent_graf.calistir("Elden Ring'in puanı kaç?", oturum="sohbet-1")
g2 = agent_graf.calistir("Peki oyuncular en çok neyi eleştiriyor?", oturum="sohbet-1")
print("1)", g1["cevap"], "\n2)", " → ".join(g2["araclar"]), "\n", g2["cevap"])

agent.iz_kaydet(sonuclar_v4, "../reports/07_test_izleri.jsonl")

talimat: 3702 | SQL kural 11: True
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm(llm)
	araclar(araclar)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm;
	araclar --> llm;
	llm -.-> __end__;
	llm -.-> araclar;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


########## [öneri] Bana iyi bir korku oyunu öner
--- ELLE: sql_calistir({"sorgu":"SELECT o.app_id, o.ad, o.olumlu_oran, o.oy_sayisi FROM oyunlar o WHERE o.olumlu_oran >= 0.85 AND o.oy_sayisi >= 100 AND NOT EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket IN ('Family Friendly')) AND EXISTS (SELECT 1 FROM oyun_etiket e WHERE e.app_id = o.app_id AND e.etiket IN ('Horror', 'Psychological Horror', 'Survival Horror', 'Gore', 'Violent', 'Blood')) AND o.yetiskin = 0 AND o.iskelet = 0 ORDER BY o.oy_sayisi DESC LIMIT 5;"})
İyi bir korku oyunu olarak önerilerim şunlar:

1. Black Myth: Wukong - %96 olu

## Adım 6 · LangGraph sürümü (src/agent_graf.py)
- Graf: START → llm → (tool_calls var mı?) → araclar → llm → … → END. Durum: mesajlar + iz (reducer operator.add: düğüm sadece yeni mesajı döndürür), model, tur. Sistem talimatı durumda tutulmaz, her LLM çağrısında başa eklenir. MemorySaver + thread_id = sohbet hafızası.
- LLM ve araçlar bizim llm.py / arac_cagir (LangChain ChatOpenAI / ToolNode yok) → iki sürüm aynı modeli aynı şekilde çağırır, karşılaştırma adil, model değişimi tek satır.
- Karşılaştırma (3 soru): aynı araçlar aynı sırada 3/3, token ≈ eşit (8.623 / 8.551 …), Sims cevabı aynı.
- Hafıza ✅: "Elden Ring'in puanı kaç?" → "Peki oyuncular en çok neyi eleştiriyor?" — oyun_bul'suz, app_id önceki mesajdan, ozetle → PC optimizasyonu, kasma, 16:9, anti-cheat.
- v4 (kural 7 sadece çocuk) ✅ elle: Wukong, Phasmophobia, Lethal Company, Dying Light, 7 Days to Die.
- Varyans: aynı soruda LangGraph çalışmasında model hatalı SQL (NOT EXISTS … NOT IN) yazdı → 0 satır → "korku oyunu yok" (kural 9 atlandı). temperature 0 ≠ deterministik → 08'de her soru 2–3 kez.
- Süre: 35 vs 5 sn = restart sonrası RAG soğuk okuma, mimari değil → API'de isinma().
- Öneri: API/Streamlit için agent_graf; agent.py anlatım ve test için.